# L45 · 声音的照片 — 声谱图生成

**目标**：把 STFT 复数矩阵转成幅度谱、功率谱、dB 谱，再用 `imshow` 画出正确标注时频轴的热力图。

🔗 Aurora 连接：[`aurora.audio.stft.magnitude_spectrogram()`](../../src/aurora/audio/stft.py) · [`aurora.audio.mel.power_to_db()`](../../src/aurora/audio/mel.py)

## 核心直觉

STFT 输出的是复数矩阵：每列是一个时间帧的频谱，每行是一个频率箱。
复数携带相位信息，但人耳感知的是**能量**，所以我们取绝对值得到幅度谱，再平方得到功率谱，再取对数压缩动态范围得到 dB 谱。
最终用 `imshow` 把这个二维矩阵渲染成热力图，横轴是时间，纵轴是频率。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine, read_wav
from aurora.audio.stft import stft, magnitude_spectrogram
from aurora.audio.mel import power_to_db

## 概念 1：幅度谱、功率谱、dB 谱

STFT 矩阵 `S` 形状为 `(n_frames, n_fft//2+1)`，每个元素是复数。

- **幅度谱**：`A = np.abs(S)`，单位是线性振幅
- **功率谱**：`P = A**2`，与能量正比
- **dB 谱**：`dB = 10 * log10(P + eps)`，eps 防 log(0)

Aurora 实现使用 eps = `1e-10`（对应 -100 dB 噪声底），并支持 `top_db` 截断防止动态范围过大。
注意：音频领域常见的另一种写法 `20 * log10(A + eps)` 与 `10 * log10(A**2 + eps)` 等价（仅在 eps 处有微小差异）。

In [ ]:
# 演示：单帧 FFT 的三种表示
SR = 16000
x = sine(440, duration=0.5, sample_rate=SR) + sine(880, duration=0.5, sample_rate=SR)

S = stft(x, n_fft=1024, hop_length=256)  # (n_frames, 513)
print(f"STFT shape: {S.shape}, dtype: {S.dtype}")

A = np.abs(S)                     # 幅度谱
P = A ** 2                        # 功率谱
dB = 10 * np.log10(P + 1e-10)    # dB 谱

print(f"幅度范围: [{A.min():.4f}, {A.max():.4f}]")
print(f"功率范围: [{P.min():.4e}, {P.max():.4e}]")
print(f"dB 范围:  [{dB.min():.1f}, {dB.max():.1f}] dB")

## 概念 2：`imshow` 的关键参数

`plt.imshow` 默认把矩阵第 0 行画在顶部（图像坐标系），但频率图习惯低频在下，所以必须设：

```
origin="lower"   # 第 0 行（DC / 0 Hz）在底部
aspect="auto"    # 不强制正方形像素，适配任意时频比例
extent=[t_min, t_max, f_min, f_max]  # 让坐标轴显示真实单位
```

`extent` 的顺序是 `[x_left, x_right, y_bottom, y_top]`，对应
`[0, duration_s, 0, sr/2]`。
`sr/2` 是奈奎斯特频率，即 STFT 能表示的最高频率。

In [ ]:
# 演示：origin 和 aspect 对图形的影响
SR = 16000
dur = 0.5
x = sine(440, duration=dur, sample_rate=SR)
S = stft(x, n_fft=1024, hop_length=256)
A = np.abs(S).T   # 转置：(频率箱, 帧数) → imshow 行=频率, 列=时间

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].imshow(A, aspect="auto")
axes[0].set_title("origin=upper (默认) — 频率倒置")

axes[1].imshow(A, origin="lower", aspect="auto",
               extent=[0, dur, 0, SR/2])
axes[1].set_title("origin=lower + extent — 正确")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylabel("频率 (Hz)")

plt.tight_layout()
plt.show()

## 概念 3：时间轴与频率轴的精确计算

**频率轴**：rfft 输出 `n_fft//2 + 1` 个箱，箱 `k` 对应频率：

```
f[k] = k * sr / n_fft      k = 0, 1, ..., n_fft//2
```

即 `f = np.arange(n_fft//2 + 1) * sr / n_fft`。

**时间轴**：第 `t` 帧的中心样本位于 `t * hop_length`（启用 `center=True` 时已经对齐），
换算成秒：

```
t[i] = i * hop_length / sr     i = 0, 1, ..., n_frames-1
```

`extent` 传入的只是边界，用 `[0, n_frames*hop/sr, 0, (n_fft//2)*sr/n_fft]` 足够精确。

In [ ]:
# 演示：手动计算轴刻度
SR = 16000
N_FFT = 1024
HOP = 256

f_bins = np.arange(N_FFT // 2 + 1) * SR / N_FFT   # 513 个频率
print(f"频率箱数: {len(f_bins)}")
print(f"频率分辨率: {SR/N_FFT:.2f} Hz/箱")
print(f"最高频率: {f_bins[-1]:.0f} Hz  (= SR/2 = {SR//2} Hz)")

x = sine(440, duration=1.0, sample_rate=SR)
S = stft(x, n_fft=N_FFT, hop_length=HOP)
n_frames = S.shape[0]
t_bins = np.arange(n_frames) * HOP / SR
print(f"\n帧数: {n_frames}")
print(f"时间分辨率: {HOP/SR*1000:.1f} ms/帧")
print(f"总时长覆盖: {t_bins[-1]:.3f} s")

## 1. ✏️ 实现 `plot_spectrogram(x, sr, win_len=1024, hop=256)`

**推理路线**：
1. 调用 `stft(x, n_fft=win_len, hop_length=hop)` 得到复数矩阵 `S`，形状 `(n_frames, win_len//2+1)`
2. `A = np.abs(S).T` 转置为 `(频率, 时间)` 布局；`dB = 20*np.log10(A + 1e-8)` 得到 dB 值
3. `plt.imshow(dB, origin="lower", aspect="auto", extent=[0, dur, 0, sr/2])` 画热力图，加 colorbar 和轴标签

**参考输入输出**：
- 输入：440 Hz + 880 Hz 双音，`sr=16000`，时长 1 s
- 期望输出：在 440 Hz 和 880 Hz 处各有一条清晰的水平亮线，其余频率为深色
- 数值检验：`dB.max()` 应在 -10 ~ 10 dB 之间（归一化正弦），`dB.min()` 约 -160 dB

<details><summary>点击查看参考实现</summary>

```python
def plot_spectrogram(x, sr, win_len=1024, hop=256):
    S = stft(x, n_fft=win_len, hop_length=hop)       # (n_frames, F)
    A = np.abs(S).T                                    # (F, n_frames)
    dB = 20 * np.log10(A + 1e-8)
    dur = len(x) / sr

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(
        dB,
        origin="lower",
        aspect="auto",
        extent=[0, dur, 0, sr / 2],
        cmap="inferno",
    )
    fig.colorbar(im, ax=ax, label="幅度 (dB)")
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("频率 (Hz)")
    ax.set_title("频谱图 (dB 幅度)")
    plt.tight_layout()
    plt.show()
```

</details>

In [ ]:
def plot_spectrogram(x, sr, win_len=1024, hop=256):
    """把信号画成 dB 幅度频谱图。

    Parameters
    ----------
    x       : 1-D float array，时域信号
    sr      : 采样率（Hz）
    win_len : FFT 窗长（样本数）
    hop     : 帧移（样本数）
    """
    # ✏️ TODO 步骤 1：调用 stft() 得到复数矩阵 S，形状 (n_frames, win_len//2+1)
    S = ...

    # ✏️ TODO 步骤 2：np.abs(S).T 得到 (F, n_frames) 幅度矩阵，再转 dB
    A = ...
    dB = ...

    dur = len(x) / sr

    # ✏️ TODO 步骤 3：imshow + colorbar + 轴标签（origin="lower", aspect="auto"）
    fig, ax = plt.subplots(figsize=(10, 4))
    ...

In [ ]:
# 目视验证：两条水平线是否清晰
SR = 16000
x_test = sine(440, duration=1.0, sample_rate=SR) + sine(880, duration=1.0, sample_rate=SR)

plot_spectrogram(x_test, SR)

# 数值检查
from aurora.audio.stft import stft as _stft
S_ref = _stft(x_test, n_fft=1024, hop_length=256)
A_ref = np.abs(S_ref).T
dB_ref = 20 * np.log10(A_ref + 1e-8)

assert dB_ref.max() > -20, "亮线 dB 值过低，检查幅度计算"
assert dB_ref.shape[0] == 513, f"频率箱数应为 513，实际 {dB_ref.shape[0]}"
print("✅ 形状正确，440 Hz / 880 Hz 水平线可见")

## 参数实验：改变 win_len 和 hop 观察时频分辨率权衡

`win_len`（窗长）和 `hop`（帧移）控制**时频分辨率的权衡**：

| 参数 | 变大时 | 代价 |
|------|--------|------|
| `win_len` 增大 | 频率分辨率提高（箱更窄） | 时间分辨率下降（帧覆盖更长） |
| `hop` 减小 | 时间分辨率提高（帧更密） | 计算量增加 |

**预期现象**：
- `win_len=256, hop=64`：水平线变粗（频率模糊），时间轴很密
- `win_len=2048, hop=512`：水平线很细（频率清晰），瞬态事件变模糊

In [ ]:
SR = 16000
x_exp = sine(440, duration=1.0, sample_rate=SR) + sine(880, duration=1.0, sample_rate=SR)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (256,  64,  "win=256 hop=64\n（时间清晰，频率模糊）"),
    (1024, 256, "win=1024 hop=256\n（均衡，默认）"),
    (2048, 512, "win=2048 hop=512\n（频率清晰，时间模糊）"),
]

for ax, (win, hop, title) in zip(axes, configs):
    S = stft(x_exp, n_fft=win, hop_length=hop)
    A = np.abs(S).T
    dB = 20 * np.log10(A + 1e-8)
    dur = len(x_exp) / SR
    im = ax.imshow(dB, origin="lower", aspect="auto",
                   extent=[0, dur, 0, SR/2], cmap="inferno",
                   vmin=dB_ref.max()-60, vmax=dB_ref.max())
    ax.set_title(title)
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("频率 (Hz)")
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
print("✅ 观察三列图中亮线宽度的变化")

## 本课小结

本课实现了 `plot_spectrogram()`，输出以 dB 为单位的时频热力图。
核心操作链为 `stft → np.abs → 平方/对数 → imshow`，其中 `origin="lower"` 和 `extent` 是正确对齐时频轴的关键。
这些操作已封装进 Aurora 的 `aurora.audio.stft.magnitude_spectrogram()` 和 `aurora.audio.mel.power_to_db()`，下节课将在此基础上构建 Mel 滤波器组，把线性频率轴压缩成听觉感知的 Mel 刻度。